# Projet Python pour la Data Science : Influence des médailles remportées par la France aux Jeux Olympiques sur le nombre le licenciés sportifs en France.

_Autrices : Melissa MIGAN, Camille PEYTHIEUX-TALDIR, Romane PLUQUET_.

## Introduction

Comme vous serez évaluées sur le fait que toutes les cellules tournent sans erreur, mettez en haut du notebook :
imports
config chemins
“seed” si besoin
un pipeline clair : collect → clean → features → viz → modèle (quand prêt)

# to do

### Sommaire

#TODO

- [Introduction](#introduction)

- [I. Création et exploration de la base de données principale](#i-création-et-exploration-de-la-base-de-données-principale)
    - [A. Récupération des données](#a-récupération-des-données)
        - [1. Médailles françaises aux Jeux Olympiques](#1-médailles-françaises-aux-jeux-olympiques)
        - [2. Licenciés sportifs en France](#2-licenciés-sportifs-en-france)
        - [3. Population départementale en France](#3-population-départementale-en-france)
    - [B. Jointure des tables](#b-jointure-des-tables)
    - [C. Contrôle de la qualité des données](#c-contrôle-de-la-qualité-des-données)
        - [1. Structure de la base](#1-structure-de-la-base)
        - [2. Valeurs manquantes](#2-valeurs-manquantes)
        - [3. Cohérence des valeurs](#3-cohérence-des-valeurs)
        - [4. Unicité des observations](#4-unicité-des-observations)
    - [D. Export de la base finale](#d-export-de-la-base-finale)

- [II. Visualisations et statistiques descriptives](#ii-visualisations-et-statistiques-descriptives)

- [Conclusion](#conclusion)

## I. Création et exploration de la base de données principale

Dans cette partie, l'objectif est d'importer et de travailler les différentes bases de données et de les joindre en une base de données exploitable. Après travail et nettoyage des données brutes (en accès public), nous utilisons trois bases de données :

| Nom de la base   | Description                                                                 | Source     | Mode d'extraction |
|-----------------|------------------------------------------------------------------------------|------------|-------------------|
| `data_medailles`  | Nombre de médailles reportées par la France aux Jeux Olympiques par sport et année (2016-2024). | Wikipedia  | Web scraping      |
| `data_licences`   | Nombre de licenciés sportifs en France par fédération, sexe, âge et département (2016-2024).       | Injep (Institut national de la jeunesse et de l'éducation populaire)          | CSV, Parquet      |
| `data_pop`        | Population départementale en France (recensements de 2016 et 2022).          | Insee      | API               |

Ces tables regroupent les données suivantes :
- La table `data_medailles` rassemble les médailles obtenues par la France aux Jeux Olympiques entre 2016 et 2024, c'est-à-dire aux JO de 2016, 2020 (qui ont eu lieu en 2021 à cause du Covid) et de 2024. 
- La table `data_licences` recense les effectif de licenciés par an entre 2016 et 2024. Elle présente également des effectifs par tranches d'âge et par genre.
- La table `data_pop` contient les populations départementales recensées en 2016 et en 2022. Elle nous permet de mener une étude des effectifs de licenciés par département, relativement à la population de ces derniers. Cette table n'étant utilisée que dans un unique graphique (dans un but de pondération), elle ne sera pas incluse dans notre base de données principale.

On importe les modules pour traiter les données, et les fonctions utilisées.

In [ ]:
%pip install -r requirements.txt

# Modules
#import os
import pandas as pd
import numpy as np
import pyarrow as pa
#import plotly.io as pio
#pio.renderers.default = "plotly_mimetype"

# Fonctions
from data import (
    gel_tableau_medailles,
    nettoyer_base,
    fusionner_bases,
    reorganiser_colonnes,
    normalisation_unicode,
    code_sport,
    code_dep,
    renommer_colonnes, 
    gel_licences,
    tableau_ratios_nr,
    melodi_extraction,
    code_dep_pop,
    clean_population,
    gel_population
)

### A. Récupération des données

#### 1. Médailles françaises aux Jeux Olympiques

Nous avons scrappé la page Wikipedia ["_France aux Jeux Olympiques_"](https://fr.wikipedia.org/wiki/France_aux_Jeux_olympiques) afin d'obtenir les tableaux des médailles (or, argent bronze) obtenues par la France lors des Jeux Olympiques, à l'aide de la fonction `tableau_scraper`.

In [ ]:
a_figer = [["or", "M.C3.A9dailles_d.27or_3"],
           ["argent", "M.C3.A9dailles_d.27argent"],
           ["bronze", "M.C3.A9dailles_de_bronze"]]

#for duo in a_figer:
    #gel_tableau_medailles(duo[0], duo[1])

 Nous avons ensuite gelé les bases scrappées dans un souci de reproductibilité, au cas où la page Wikipédia soit modifiée. Nous avons ainsi obtenu trois tables :
- `data_or` : renseignant le nombre de médailles d'or obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_argent` : renseignant le nombre de médailles d'argent obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_bronze` : renseignant le nombre de médailles de bronze obtenues par la France aux Jeux Olympiques, par sport et par année.

In [ ]:
data_or = pd.read_csv(f"data/data_brut/data_or_jo.csv")
data_argent = pd.read_csv(f"data/data_brut/data_argent_jo.csv")
data_bronze = pd.read_csv(f"data/data_brut/data_bronze_jo.csv")

display(data_or.head())
display(data_argent.head())
display(data_bronze.head())

Nous avons ensuite nettoyé ces tables :
- nous avons supprimé les lignes et colonnes vides, ainsi que la colonne `Place`, qui ne servait que lors du scraping (pour être sûr de bien scraper toutes les lignes) ;
- nous avons retiré les années hors de notre période d'analyse (2016-2024) ;
- nous avons supprimé la colonne de total des médailles.

In [ ]:
data_or_clean = nettoyer_base(data_or)
data_argent_clean = nettoyer_base(data_argent)
data_bronze_clean = nettoyer_base(data_bronze)

display(data_or_clean.head())
display(data_argent_clean.head())
display(data_bronze_clean.head())

Suite à cela, nous avons joint ces trois tables afin d'obtenir la table `data_medailles_jo`, renseignant le nombre de médailles (toutes couleurs confondues) obtenues par la France aux Jeux Olympiques de 2016, 2020 et 2024.

Nous y avons ajouté trois variables :
- `code_sport`, un code permettant plus tard la jointure avec la table des licenciés sportifs en France,
- `total_medailles_2020`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2020,
- `total_medailles_2024`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2024.

In [ ]:
data_medailles = fusionner_bases(data_or, data_argent, data_bronze)
display(data_medailles.sample(5))

#### 2. Licenciés sportifs en France

Nous avons exploité les données de licenciés en France fournies par l'Injep, l'Institut national de la Jeunesse et de l'Education populaire. Les données se présentent sous forme brute comme un fichier CSV par an. Cependant, ces fichiers étant trop lourds pour être importés directement dans Git, nous optons pour une gestion des données par fichiers au format parquet. 


##### a. Construction de la base 

Notre but ici est de contruire une base de données au format long. Chaque base disposant déjà d'une colonne année, nous devons alors les concaténer pour obtenir la base au format désiré. Pour que l'opération se déroule correctement, nous réorganisons dans un premier temps les colonnes de chacune des bases, de sorte que chacune ait les mêmes colonnes dans le même ordre.

In [1]:
liste_fichiers = ["Lics_2016_semidef.parquet",
                  "Lics_2017_semidef.parquet", 
                  "Lics_2018_semidef.parquet",
                  "Lics_2019_def.parquet",
                  "Lics_2020_def.parquet",
                  "Lics_2021_def.parquet",
                  "Lics_2022_def.parquet",
                  "Lics_2023_semidef.parquet",
                  "Lics_2024_semidef.parquet"]

data_licences = reorganiser_colonnes(liste_fichiers)
data_licences = pa.concat_tables(data_licences)

NameError: name 'reorganiser_colonnes' is not defined

Afin d'éviter les problèmes de sélection de données, car nous disposons de variables dont les modalités sont textuelles, nous normalisons tous les caractères avec la norme unicode. Nous ajoutons ensuite plusieurs variables permettant un niveau d'analyse plus général que celui très fin proposé par la base de données ainsi qu'une harmonisation entre les différentes bases utilisées : 
- `code_sport` : catégorise les fédérations selon le sport pratiqué. Nous choisissons de catégoriser en "divers" (`code_sport` : DIV) les fédérations qui pratiquent un sport non-olympique. Ce code est le même que celui ajouté à la table des médailles. 
- `code_dep` : indique le département par son numéro seulement. 

In [ ]:
data_licences = normalisation_unicode(data_licences)
data_licences = code_sport(data_licences)
data_licences = code_dep(data_licences, "Département")

Nous renommons ensuite les colonnes pour avoir des noms de variable sans majusucles ni accents. Nous ne gardons que les colonnes qui nous seront utiles pour la suite, à savoir celles concernant le nom de la fédération, l'année de recensement des licences, le sexe des licencié.e.s, les tranches d'âge (age, tranches fines et grandes fines), le nombre de licences annuelles, le code sport et le code département. Finalement, nous gelons la table dans un fichier parquet pour la réutiliser par la suite telle que construite ici. 

In [ ]:
data_licences = renommer_colonnes(data_licences)
data_licences = data_licences[["federation","annee", "sexe", "age", "tranche_age","grande_tranche_age","licences_annuelles","code_sport","code_dep"]]
#gel_licences(data_licences)
display(data_licences.sample(5))
display(data_licences["federation"].describe())

Nous obtenons une base de données comprenant plus de 7,35 millions de lignes. 118 fédérations sportives y sont recensées, et ce sur neuf années, de 2016 à 2024. La variable `code_sport` nous permet de catégoriser facilement les fédérations selon le sport pratiqué. Nous choisissons la modalité "DIV" pour les sport non-olympiques. Ainsi, 33 sports olympiques sont présents dans notre base de données. 

Nous disposons de plusieurs variables permettant de distinguer les licenciés selon des critères socio-démographiques. 
- trois variables d'âge :  
    - la variable `age` qui donne l'âge précis des licenciés, 
    - la variable `tranche_age` qui répertorie les licenciés selon 18 tranches d'âges "fines", par exemple ceux ayant de 10 à 14 ans,
    - la variable `grande_tranche_age` qui classe les licenciés selon cinq tranches d'âge plus larges, comme par exemple la catégorie Adultes (21-55 ans),
- la variable `sexe`, qui présente deux modalités, H ou F, et permet de classer les licenciés selon leur genre,
- la variable `code_dep`, qui renseigne le département dans lequel sont enregistrés les licenciés.

Finalement, notre variable d'intérêt est la variable `licences_annuelles`, que nous pouvons agréger selon toutes les variables présentes dans la base, en prenant les précautions nécessaires, détaillées dans la partie suivante.

##### b. Précautions : comparabilité dans le temps

Nos données de licences proviennent de fichiers distincts pour chaque année recensée. Ces fichiers sont de deux types :
- `semidef` pour les années 2016 à 2018 et 2023 à 2024, qui n'ont pas encore été "géocodées",
- `def` pour les années 2019 à 2022, qui sont "géocodées".

La documentation de ces données affirme que les données sont comparables dans le temps à condition d'être agrégées par fédération. Ainsi, comme notre code sport est directement dérivé des fédérations, l'agrégation selon la variable `code_sport` ne posera pas de problème de compabilité. Cependant, il est précisé que les "données par sexe, et/ou par âge, et/ou par département/région ne doivent pas être comparées dans le temps directement". Il faut dans ce cas là bien prendre en compte les effectifs non répartis (NR), c'est-à-dire qui n'ont pas pu être classés selon un département, une catégorie d'âge ou de genre, afin d'obtenir des résultats comparables dans le temps. 

Pour d'avoir une idée de l'ampleur de la non répartition géographique des effectifs de licences dans les données, nous calculons les ratios d'effectifs de licences géographiquement non répartis sur la totalité des effectifs de licences par an, ainsi que dans la base regroupant toutes les années. 

In [ ]:
display(tableau_ratios_nr(data_licences, "code_dep"))

On constate alors que la proportion de licences non géographiquement réparties est bien plus importante pour les premiers fichiers `semidef`, c'est-à-dire de 2016 à 2018, alors que par la suite, cette proportion n'excède pas les 1%. Ainsi, dès que nous mènerons une analyse géographique, nous prendrons en compte dans l'élaboration et l'analyse des résultats le fait que, de 2016 à 2018, autour de 5% des effectifs de licences ne sont géographiquement pas attribués, dans la mesure où cette proportion est significativement importante. Nous montrons ensuite que, comme le suggère la précision dans la documentation sur le fait que certains fichiers soient "géocodés" ou non, la non répartition géographique est la plus importante dans notre jeu de données, grâce à un calcul de ratio similaire. 

In [ ]:
display(tableau_ratios_nr(data_licences, "sexe"))
display(tableau_ratios_nr(data_licences, "age"))
display(tableau_ratios_nr(data_licences, "tranche_age"))
display(tableau_ratios_nr(data_licences, "grande_tranche_age"))

En ce qui concerne l'âge et le sexe, on constate que la non répartition est globalement moins importante, mais toujours assez marquée de 2016 à 2018. La non répartition pour les variables d'âge en tranches est strictement égale à celle de la variable d'âge, puisque ces dernières découlent directement de la première. La non répartition en terme d'âge n'excède pas les 2% par an, et tend vers de très faibles valeurs à partir de 2019 (< 0,31%). La non répartition par sexe, elle, n'excède pas les 2,43% et est nulle de 2020 à 2024. Ainsi, bien que ce phénomène soit moins important pour l'âge et le sexe, nous le prendrons en compte dans nos analyses par âge et par sexe. 

#### 3. Population départementale en France

Dans l'optique de mener une analyse des effectifs de licenciés par départements, proportionnellement à leur population, nous récupérons les données de population départementale en France grâce à l'API Melodi de l'INSEE. Nous choisissons d'utiliser le jeu de données de la population de municipale et non de la population de référence, car la population municipale est comptabilisée tous les ans, tout comme nos données de licences. Ce choix permet une cohérence temporelle entre nos données. Par ailleurs, d'après l'INSEE, la population municipale est "désormais [...] la notion de population la plus utilisée en statistiques".

La documentation de l'API fournit directement le code permettant d'extraire les données voulues, que nous reprenons dans la fonction permettant l'extraction des données. L'URL d'extraction a été obtenu par visualisation de la base de données au niveau de précision désiré, c'est-à-dire au niveau départemental ; et permet de sélectionner directement les années d'intérêt (2016 à 2023). 

La base de données extraite comporte 800 lignes : la population pour les 96 départements de France métropolitaines et quatre départements d'Outre-mer (Guadeloupe, Martinique, Guyane, La Réunion), sur huit années. Les données n'étant pas encore disponibles pour l'année 2024, nous utiliserons celles de 2023 pour des analyses relatives à la population en 2024.

In [ ]:
url_api = "https://api.insee.fr/melodi/data/DS_POPULATIONS_HISTORIQUES?TIME_PERIOD=2016&TIME_PERIOD=2017&TIME_PERIOD=2018&TIME_PERIOD=2019&TIME_PERIOD=2020&TIME_PERIOD=2021&TIME_PERIOD=2022&TIME_PERIOD=2023&GEO=DEP"
data_pop = melodi_extraction(url_api)
display(data_pop.sample(5))

Nous ne sélectionnons que les variables utiles pour notre analyse, c'est-à-dire celles correspondant au département, à l'année et à la population comptablisée dans le département, puisque les variables de fréquence de mesure et de type de mesure sont unimodales. Nous renommons également les colonnes avec le même format que pour les bases présentées précédemment. Finalement, nous ajoutons le même code département que dans les autres bases, à partir de la variable de département. 

In [ ]:
display(data_pop["FREQ"].unique())
display(data_pop["POPREF_MEASURE"].unique())

In [ ]:
data_pop = data_pop[["GEO", "TIME_PERIOD", "OBS_VALUE_NIVEAU"]]
data_pop.columns = ["departement", "annee", "population"]
data_pop = code_dep_pop(data_pop, "departement")

Nous nettoyons ensuite cette base, en forçant les types des variables (année en entiers, département en variables textuelles), en supprimant les départements non cartographiables et en les triant par ordre croissant. Nous gelons ensuite les données dans un fichier CSV. 

In [ ]:
data_pop_clean = clean_population(data_pop)
#gel_population(data_pop_clean)
data_pop_clean.sample(5)

### B. Jointure des tables

Nous avons joint les tables médailles et licences pour pouvoir étudier en détail l'effet de remporter des médailles aux Jeux Olympiques sur l'évolution du nombre de licenciés sportifs. Nous avons joint par la gauche en utilisant la clé `code_sport` pour ne pas démultiplier le nombre de lignes dans notre DataFrame : nous avons un unique code sport par ligne dans notre table médailles.

In [ ]:
data_complet = pd.merge(data_licences, data_medailles, how='left', on="code_sport")
data_complet.sample(5)
#data_complet["code_sport"].unique()

### C. Contrôle de la qualité des données

#### 1. Structure de la base

In [ ]:
data_complet.shape

In [ ]:
data_complet.dtypes

La base de données contient plus de sept millions d’observations et décrit les licences sportives selon l’année, le département du club sportif, le sport, le sexe, l’âge, et les médailles remportées aux Jeux Olympiques selon les sports. Les types des variables sont cohérents avec leur interprétation. Puisque pandas reconnait les _strings_ comme des _objects_, il est normal que certaines colonnes (comme `sexe` par exemple) soient indiquées être des objets alors qu'elles sont censées être des chaînes de caractères.

#### 2. Valeurs manquantes

In [ ]:
na_table = (
    data_complet.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("taux_manquants")
)

na_table

Les valeurs manquantes concernent principalement les données issues de la tables `data_medailles`, et sont liées à la jointure par la gauche. Les valeurs manquantes concernant la variable géographique (`code_dep`, soit les départements) sont causées par les clubs sportifs de l'étranger et par les départements non répartis.

In [ ]:
data_complet[data_complet["code_dep"].isna()]

#### 3. Cohérence des valeurs

##### a. Années observées

In [ ]:
data_complet["annee"].min(), data_complet["annee"].max()

Les années couvertes par la base (2016-2024) sont cohérentes avec le périmètre temporel de l’étude.

##### b. Effectifs de licenciés

In [ ]:
data_complet["licences_annuelles"].describe()

In [ ]:
(data_complet["licences_annuelles"] < 0).sum()

Aucun effectif négatif n’est observé. Les ordres de grandeur des effectifs sont cohérents avec des données de licences sportives.

#### 4. Unicité des observations

In [ ]:
data_complet.duplicated().sum()

Ces doublons stricts représentent environ 0,03% de notre base de données, soit une proportion négligeable. Nous les supprimons afin de garantir l'unicité des observations, sans impact significatif sur les analyses.

In [ ]:
data_complet = data_complet.drop_duplicates()

### D. Export de la base finale

Nous avons ensuite exporté cette table dans le dossier `data_clean`, lui même dans le dossier `data`.

## TODO

In [ ]:
#data_complet.to_parquet("data_complet.parquet")


#OUTPUT_DIR.mkdir(exist_ok=True)

#musees.to_csv(OUTPUT_DIR / "musees.csv", index=False)
#frequentation_annuelle.to_csv(OUTPUT_DIR / "frequentation_annuelle.csv", index=False)
#freq_excel_long.to_csv(OUTPUT_DIR / "frequentation_excel_long.csv", index=False)
#df_modele_clean.to_csv(OUTPUT_DIR / "df_modele_musees.csv", index=False)

#print("Fichiers exportés dans :", OUTPUT_DIR.resolve())


## II. Visualisations et statistiques descriptives

In [ ]:
from fonctions_graphiques import (
    charger_donnees,
    widget_licences_par_sport,
    widget_graphique_licences_et_medailles,
    classement_sports_medailles,
    croissance_licencies_post_jo,
    widgets_evolution_licencies,
    widgets_evolution_licences_tranches_grande_age
)

In [ ]:
(data_complet_2, gdf_dep, pop) = charger_donnees(
    data_complet_path="data/data_complet.parquet",
    geojson_path="departements.geojson",
    population_path="data/data_population/population_dept.csv",
)

### A(?). Exploration "naïve" de la base

In [ ]:
widget_licences_par_sport(data_complet)

Nous pouvons remarquer une baisse marquée du nombre de licenciés sportifs en 2021, et ce dans presque tous les sports - sauf golf, pentathlon moderne, surf, tir, voilee et équitation. Cette baisse est liée à la pandémie de Covid 19 et aux mesures sanitaires prises en France, qui ont fortement restreint la pratique sportive. Il nous faudra inclure cet évènement exceptionnel dans nos analyses et modélisations.

### A. Medailles et licenciés

Nous essayons ici d'observer les conséquences immédiates du nombre de médailles remportées aux Jeux Olympiques sur le nombre de licenciés sportifs.

Dans un premier temps, il s'agit d'observer quels sports ont remporté le plus de médailles olympiques grâce à la fonction `classement_sports_medailles`.

In [ ]:
classement_sports_medailles(data_complet, 'all')

commentaire : TODO

Les Jeux Olympiques de 2020 ayant eu lieu en 2021 en raison de la pandémie de Covid 19, nous avons décidé de définir l'année des médailles des JO 2020 en 2021 sur les graphiques.

In [ ]:
widget_graphique_licences_et_medailles(data_complet)

Nous observons que l'obtention de médailles olympiques, notamment en or, affecte visiblement le nombre de licenciés sportifs dans certaines disciplines (volley ball, rugby, handball...). Cependant, cette observation n'est pas vraie pour tous les sports (judo, tir à l'arc...). Pour comprendre ces différences, nous nous intéresseront plus en détail aux caractéristiques des vainqueurs olympiques et des licenciés : sexe, âge.

Nous tentons maintenant de voir quels sports ont été le plus affectés par les Jeux Olympiques : lesquels ont-ils le plus vu leur nombre de licenciés augmenter dans les deux années suivant les Jeux Olympiques ? Nous choisissons de voir ce taux de croissance à t+2 pour éviter au maximum de percevoir l'effet du Covid 19.

In [ ]:
display(croissance_licencies_post_jo(data_complet, 2016, 2).head())
display(croissance_licencies_post_jo(data_complet, 2020, 2).head())

### B. Evolution du nombre de licenciés

In [ ]:
widgets_evolution_licencies(data_complet, gdf_dep)

In [ ]:
widgets_evolution_licences_tranches_grande_age(data_complet)